<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 48px 40px; border-radius: 12px; color: white; font-family: 'Segoe UI', sans-serif;">
  <h1 style="margin:0 0 8px 0; font-size:2.2rem; font-weight:700; letter-spacing:-0.5px;">
    Classic DNS Tunneling Detection
  </h1>
  <h2 style="margin:0 0 24px 0; font-size:1.1rem; font-weight:400; opacity:0.75;">
    UDP/53 and TCP/53 machine-learning detector for live SOC-style monitoring
  </h2>
  <hr style="border-color:rgba(255,255,255,0.2); margin-bottom:24px;">
  <table style="color:white; font-size:0.92rem; border-collapse:collapse; width:100%;">
    <tr>
      <td style="padding:4px 24px 4px 0; opacity:0.65; white-space:nowrap;">Dataset</td>
      <td><strong>PCAP-extracted classic DNS flows</strong> from DNS tunneling captures and benign resolver traffic</td>
    </tr>
    <tr>
      <td style="padding:4px 24px 4px 0; opacity:0.65;">Task</td>
      <td>Binary classification: benign classic DNS vs. DNS tunneling behavior</td>
    </tr>
    <tr>
      <td style="padding:4px 24px 4px 0; opacity:0.65;">Transports</td>
      <td>UDP53 and TCP53 use the unified classic detector; DoH is handled separately by <code>artifacts/doh/</code></td>
    </tr>
    <tr>
      <td style="padding:4px 24px 4px 0; opacity:0.65;">Models</td>
      <td>Random Forest · XGBoost · Logistic Regression</td>
    </tr>
    <tr>
      <td style="padding:4px 24px 4px 0; opacity:0.65;">Deployment</td>
      <td><code>artifacts/classic_dns/best_model.pkl</code> loaded by <code>pipeline/tui.py</code> with a 70% malicious alert threshold</td>
    </tr>
  </table>
</div>

---
## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Environment Setup](#1) | Imports, path detection, global configuration |
| 2 | [Dataset Loading](#2) | Load the PCAP-extracted classic DNS dataset safely |
| 3 | [Data Quality And EDA](#3) | Missing values, class balance, transport coverage, numeric ranges |
| 4 | [Feature Engineering](#4) | DNS lexical, timing, payload, and TCP interaction features |
| 5 | [Train/Test Split](#5) | Group-aware split to reduce PCAP source leakage |
| 6 | [Model Training](#6) | Train Logistic Regression, Random Forest, and XGBoost |
| 7 | [Threshold Review](#7) | Evaluate the 70% TUI malicious announcement threshold |
| 8 | [Known-Good Sanity Check](#8) | Validate normal infrastructure lookups such as `ping.archlinux.org` |
| 9 | [Artifact Export](#9) | Save models, metrics, feature names, and metadata |
| 10 | [TUI Deployment Mapping](#10) | Confirm UDP53/TCP53/DoH routing used by the demo monitor |

---

<a id="1"></a>
## 1. Environment Setup

This cell imports the libraries, fixes the random seed, and resolves paths from the project root. It works whether Jupyter starts in the repository root or inside the `notebooks/` folder.

In [3]:
# ── Environment setup ────────────────────────────────────────────────────────
import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

TARGET = "label"
RANDOM_STATE = 42
DECISION_THRESHOLD = 0.70

# Resolve the project root whether Jupyter starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pipeline").exists() and (candidate / "datasets").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not locate project root. Start Jupyter from the repository root "
        "or keep the datasets/ and pipeline/ folders in this project."
    )

CFG = {
    "dataset": PROJECT_ROOT / "datasets" / "classic_dns_from_pcaps.csv",
    "output_dir": PROJECT_ROOT / "artifacts" / "classic_dns",
    "decision_threshold": DECISION_THRESHOLD,
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
}

DATASET = CFG["dataset"]
OUTPUT_DIR = CFG["output_dir"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLUMNS = {
    TARGET,
    "source_file",
    "top_level",
    "split_hint",
    "transport",
    "src_ip",
    "dst_ip",
    "timestamp",
    "query_name",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET}")
print(f"Artifact output: {OUTPUT_DIR}")

Project root: /home/gyro/Documents/Projects/cns_projects/c1-dns-tunneling
Dataset path: /home/gyro/Documents/Projects/cns_projects/c1-dns-tunneling/datasets/classic_dns_from_pcaps.csv
Artifact output: /home/gyro/Documents/Projects/cns_projects/c1-dns-tunneling/artifacts/classic_dns


<a id="2"></a>
## 2. Dataset Loading

The classic DNS dataset is stored in `datasets/classic_dns_from_pcaps.csv`. Because notebooks are often launched from either the repository root or the `notebooks/` folder, the setup cell resolves `PROJECT_ROOT` automatically and builds absolute paths for the dataset and artifact directory.

If this cell reports that the dataset is missing, regenerate it with:

```bash
.venv/bin/python pipeline/extract_classic_dns_pcaps.py \
  --dataset-root DNS-Tunnel-Datasets \
  --output datasets/classic_dns_from_pcaps.csv
```

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    path = Path(path).expanduser().resolve()
    if not path.exists():
        message = (
            f"Dataset not found: {path}\n\n"
            "Expected file: datasets/classic_dns_from_pcaps.csv\n"
            "From the project root, regenerate it with:\n"
            ".venv/bin/python pipeline/extract_classic_dns_pcaps.py "
            "--dataset-root DNS-Tunnel-Datasets "
            "--output datasets/classic_dns_from_pcaps.csv"
        )
        raise FileNotFoundError(message)
    df = pd.read_csv(path)
    if TARGET not in df.columns:
        raise ValueError(f"Dataset must contain {TARGET!r}")
    if df.empty:
        raise ValueError("Dataset is empty")

    before = len(df)
    df = df.drop_duplicates()
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.dropna(subset=[TARGET])
    df[TARGET] = df[TARGET].astype(int)
    unexpected = set(df[TARGET].unique()) - {0, 1}
    if unexpected:
        raise ValueError(f"Unexpected labels: {unexpected}")

    print(f"Loaded dataset: {path}")
    print(f"Loaded rows: {len(df):,}")
    print(f"Dropped duplicates / bad labels: {before - len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    return df

df = load_dataset(DATASET)
pd.DataFrame({
    "label": df[TARGET].value_counts().sort_index(),
    "percent": df[TARGET].value_counts(normalize=True).sort_index().mul(100).round(2),
})

Loaded dataset: /home/gyro/Documents/Projects/cns_projects/c1-dns-tunneling/datasets/classic_dns_from_pcaps.csv
Loaded rows: 577,559
Dropped duplicates / bad labels: 4
Columns: 52


,label,percent
label,,
0,302045,52.3
1,275514,47.7


<a id="3"></a>
## 3. Data Quality And Exploratory Analysis

Before training, we verify that the table is usable for modeling:

| Check | Why it matters |
|---|---|
| Label balance | Confirms both benign and tunneling traffic are represented |
| Transport coverage | Confirms which classic transports are present in the extracted PCAPs |
| Missing values | Reveals parser gaps or optional TCP-only fields |
| Numeric ranges | Finds impossible values, extreme timing, or payload anomalies |
| Source grouping | Helps avoid evaluating on flows from the same capture used for training |

In [5]:
missing = df.isna().mean().sort_values(ascending=False).head(15).rename("missing_fraction")
transport_counts = df["transport"].value_counts(dropna=False) if "transport" in df else pd.Series(dtype=int)
label_transport = pd.crosstab(df.get("transport", pd.Series("unknown", index=df.index)), df[TARGET], normalize="index").round(4)

numeric_summary = df.select_dtypes(include=[np.number]).describe().T[["mean", "std", "min", "50%", "max"]].round(3)

print("Transport counts:")
print(transport_counts.to_string())
print("\nLabel ratio by transport:")
print(label_transport.to_string())
print("\nTop missing columns:")
print(missing.to_string())

numeric_summary.head(20)

Transport counts:
transport
UDP53    577559

Label ratio by transport:
label          0      1
transport              
UDP53      0.523  0.477

Top missing columns:
source_file        0.0
top_level          0.0
split_hint         0.0
label              0.0
transport          0.0
transport_code     0.0
src_ip             0.0
dst_ip             0.0
timestamp          0.0
dns_id             0.0
query_name         0.0
query_type         0.0
response_type      0.0
query_length       0.0
response_length    0.0


,mean,std,min,50%,max
label,0.477,0.499,0.0,0.000,1.000
transport_code,0.000,0.000,0.0,0.000,0.000
dns_id,0.000,0.000,0.0,0.000,0.000
query_type,5.326,7.296,1.0,1.000,65.000
response_type,5.604,7.037,0.0,1.000,28.000
query_length,75.873,94.183,4.0,22.000,252.000
response_length,180.451,199.594,0.0,92.000,1471.000
payload_size,276.695,263.381,23.0,124.000,1753.000
ttl,2645.368,11178.305,0.0,60.000,86400.000
answer_count,1.277,1.138,0.0,1.000,30.000


<a id="4"></a>
## 4. Feature Engineering

DNS tunneling changes both the content and rhythm of DNS traffic. The model uses feature families that are meaningful for classic DNS inspection:

| Feature family | Examples | Security interpretation |
|---|---|---|
| Lexical query shape | query length, entropy, label length, digit/hex ratio | Encoded payloads often look longer and more random than normal hostnames |
| Record behavior | TXT/NULL flags, answer count, response type | Tunnels often use records that can carry arbitrary data |
| Timing and rate | query rate, inter-arrival time | Automated tunnels produce repeated, machine-like query bursts |
| Payload pressure | payload/query ratios, response/query ratios | Encapsulation changes normal DNS size relationships |
| TCP DNS behavior | segment count, DNS length field, retransmission pressure | TCP-based tunnels create larger and more persistent DNS exchanges |

In [6]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.replace(0, np.nan)

_OPTIONAL_COLUMNS = [
    "tcp_payload_size", "dns_length_field", "segment_count",
    "retransmission_ratio", "message_count", "max_message_size",
    "avg_message_size", "rcode",
]

def add_derived_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    numeric_candidates = [column for column in frame.columns if column not in DROP_COLUMNS]
    for column in numeric_candidates:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    for column in _OPTIONAL_COLUMNS:
        if column not in frame.columns:
            frame[column] = 0

    frame["response_to_query_ratio_len"] = safe_divide(frame["response_length"], frame["query_length"])
    frame["payload_to_query_ratio"] = safe_divide(frame["payload_size"], frame["query_length"])
    frame["payload_to_response_ratio"] = safe_divide(frame["payload_size"], frame["response_length"])
    frame["entropy_per_query_char"] = safe_divide(frame["query_entropy"], frame["query_length"])
    frame["label_entropy_per_max_label"] = safe_divide(frame["label_entropy"], frame["max_label_length"])
    frame["chars_per_label"] = safe_divide(frame["unique_chars"], frame["max_label_length"])
    frame["query_rate_per_iat"] = safe_divide(frame["query_rate"], frame["inter_arrival_time"])
    frame["high_entropy_long_query"] = frame["query_entropy"] * frame["query_length"]
    frame["digit_hex_mix"] = frame["digit_ratio"] * frame["hex_ratio"]
    frame["txt_or_null_record"] = ((frame["has_txt_record"].fillna(0) > 0) | (frame["has_null_record"].fillna(0) > 0)).astype(int)
    frame["tcp_payload_to_dns_payload_ratio"] = safe_divide(frame["tcp_payload_size"], frame["payload_size"])
    frame["dns_length_to_tcp_payload_ratio"] = safe_divide(frame["dns_length_field"], frame["tcp_payload_size"])
    frame["segments_per_message"] = safe_divide(frame["segment_count"], frame["message_count"])
    frame["payload_per_segment"] = safe_divide(frame["tcp_payload_size"], frame["segment_count"])
    frame["message_size_spread"] = frame["max_message_size"] - frame["avg_message_size"]
    frame["retransmission_payload_pressure"] = frame["retransmission_ratio"] * frame["payload_size"]
    frame["nxdomain_flag"] = (frame["rcode"].fillna(0) == 3).astype(int)
    return frame.replace([np.inf, -np.inf], np.nan)

def feature_columns(frame: pd.DataFrame) -> list[str]:
    columns = [
        column
        for column in frame.columns
        if column not in DROP_COLUMNS and pd.api.types.is_numeric_dtype(frame[column])
    ]
    return [column for column in columns if frame[column].notna().any()]

df = add_derived_features(df)
features = feature_columns(df)
print(f"Feature count: {len(features)}")
features[:15]

Feature count: 58


['transport_code',
 'dns_id',
 'query_type',
 'response_type',
 'query_length',
 'response_length',
 'payload_size',
 'ttl',
 'answer_count',
 'additional_count',
 'has_txt_record',
 'has_null_record',
 'is_recursive',
 'truncated_flag',
 'authoritative_flag']

<a id="5"></a>
## 5. Train/Test Split

When the `source_file` column has enough unique capture groups, the notebook uses a group-aware split. This is important because packets from the same PCAP can share strong capture-specific patterns; group splitting gives a more honest estimate than randomly mixing flows from the same source into train and test.

In [7]:
def split_data(frame: pd.DataFrame, features: list[str]):
    X = frame[features]
    y = frame[TARGET]
    groups = frame["source_file"] if "source_file" in frame.columns else None
    if groups is not None and groups.nunique() >= 4:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
        train_index, test_index = next(splitter.split(X, y, groups))
        return X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = split_data(df, features)
print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Train positive ratio: {y_train.mean():.3f}")
print(f"Test positive ratio: {y_test.mean():.3f}")

Train rows: 459,633
Test rows: 117,926
Train positive ratio: 0.507
Test positive ratio: 0.360


<a id="6"></a>
## 6. Model Training

Three model families are trained for comparison:

| Model | Why it is included |
|---|---|
| Logistic Regression | Fast, interpretable linear baseline after scaling |
| Random Forest | Strong tabular baseline that handles nonlinear feature interactions well |
| XGBoost | Gradient-boosted trees for high-performing structured-data classification |

The best model is selected by F1 score, then recall and ROC-AUC as tie-breakers, and exported as `best_model.pkl` for the TUI.

In [8]:
def build_models(features: list[str]) -> dict[str, Pipeline]:
    scaled_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                features,
            )
        ],
        remainder="drop",
    )
    tree_preprocessor = ColumnTransformer(
        transformers=[("numeric", Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]), features)],
        remainder="drop",
    )
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("preprocess", scaled_preprocessor),
                ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
            ]
        ),
        "random_forest": Pipeline(
            steps=[
                ("preprocess", tree_preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=350,
                        class_weight="balanced",
                        min_samples_leaf=2,
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "xgboost": Pipeline(
            steps=[
                ("preprocess", tree_preprocessor),
                (
                    "model",
                    XGBClassifier(
                        n_estimators=350,
                        max_depth=4,
                        learning_rate=0.05,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        objective="binary:logistic",
                        eval_metric="logloss",
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }

def evaluate(model: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> tuple[dict[str, Any], list[list[int]]]:
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba) if len(set(y_test)) > 1 else 0.0,
        "avg_precision": average_precision_score(y_test, proba) if len(set(y_test)) > 1 else 0.0,
        "classification_report": classification_report(y_test, pred, output_dict=True, zero_division=0),
    }
    return metrics, confusion_matrix(y_test, pred).tolist()

models = build_models(features)
metrics_rows = []
full_metrics = {}
matrices = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    metrics, matrix = evaluate(model, X_test, y_test)
    joblib.dump(model, OUTPUT_DIR / f"{name}.pkl")
    metrics_rows.append({key: metrics[key] for key in ["accuracy", "precision", "recall", "f1", "roc_auc", "avg_precision"]} | {"model": name})
    full_metrics[name] = metrics
    matrices[name] = matrix

metrics_df = pd.DataFrame(metrics_rows).sort_values(by=["f1", "recall", "roc_auc"], ascending=False)
best_model_name = str(metrics_df.iloc[0]["model"])
best_model = models[best_model_name]
joblib.dump(best_model, OUTPUT_DIR / "best_model.pkl")
metrics_df

,accuracy,precision,recall,f1,roc_auc,avg_precision,model
1,1.000000,1.000000,1.000000,1.000000,1.0,1.0,random_forest
2,0.999966,0.999976,0.999929,0.999953,1.0,1.0,xgboost
0,0.999907,0.999953,0.999788,0.999870,1.0,1.0,logistic_regression


<a id="7"></a>
## 7. Threshold Review For TUI Announcements

The model outputs a malicious probability for each flow. The TUI announces a flow as malicious only when that probability is at least `0.70`. This keeps borderline normal DNS lookups from becoming class-demo alerts while still showing the exact probability in the flow detail panel.

In [9]:
def evaluate_at_threshold(proba: np.ndarray, y_test: pd.Series, threshold: float) -> dict[str, Any]:
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "false_positive_rate": fp / max(fp + tn, 1),
        "false_negative_rate": fn / max(fn + tp, 1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

best_proba = best_model.predict_proba(X_test)[:, 1]
threshold_df = pd.DataFrame(
    evaluate_at_threshold(best_proba, y_test, threshold)
    for threshold in np.round(np.arange(0.05, 1.0, 0.05), 2)
)
selected_threshold = threshold_df.loc[(threshold_df["threshold"] - DECISION_THRESHOLD).abs().idxmin()]
selected_threshold_metrics = {
    key: (int(value) if key in {"tn", "fp", "fn", "tp"} else float(value))
    for key, value in selected_threshold.to_dict().items()
}
threshold_df.to_csv(OUTPUT_DIR / "threshold_analysis.csv", index=False)
threshold_df.loc[threshold_df["threshold"].between(0.50, 0.90)].reset_index(drop=True)

,threshold,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,tn,fp,fn,tp
0,0.50,1.000000,1.0,1.000000,1.000000,0.0,0.000000,75486,0,0,42440
1,0.55,1.000000,1.0,1.000000,1.000000,0.0,0.000000,75486,0,0,42440
2,0.60,1.000000,1.0,1.000000,1.000000,0.0,0.000000,75486,0,0,42440
3,0.65,1.000000,1.0,1.000000,1.000000,0.0,0.000000,75486,0,0,42440
4,0.70,0.999992,1.0,0.999976,0.999988,0.0,0.000024,75486,0,1,42439
5,0.75,0.998313,1.0,0.995311,0.997650,0.0,0.004689,75486,0,199,42241
6,0.80,0.997931,1.0,0.994251,0.997117,0.0,0.005749,75486,0,244,42196
7,0.85,0.973187,1.0,0.925495,0.961306,0.0,0.074505,75486,0,3162,39278
8,0.90,0.909876,1.0,0.749576,0.856866,0.0,0.250424,75486,0,10628,31812


<a id="8"></a>
## 8. Known-Good Infrastructure Sanity Check

This section checks realistic, ordinary infrastructure lookups such as `ping.archlinux.org`, mirrors, package repositories, and NTP pools. These should remain benign with the 70% TUI threshold because they are single, low-rate DNS lookups with normal record types.

In [10]:
def _entropy(text: str) -> float:
    if not text:
        return 0.0
    counts = pd.Series(list(text)).value_counts(normalize=True)
    return float(-(counts * np.log2(counts)).sum())

def normal_lookup_row(query_name: str, qtype: int = 1) -> dict[str, object]:
    labels = [label for label in query_name.rstrip(".").split(".") if label]
    compact = "".join(labels).lower()
    hex_chars = set("0123456789abcdef")
    consonants = set("bcdfghjklmnpqrstvwxyz")
    return {
        "source_file": "sanity_known_good",
        "top_level": labels[-1] if labels else "",
        "split_hint": "sanity",
        "transport": "UDP53",
        "src_ip": "192.0.2.10",
        "dst_ip": "9.9.9.9",
        "timestamp": "2026-05-08 12:00:00",
        "query_name": query_name,
        "label": 0,
        "query_length": len(query_name),
        "response_length": 84,
        "payload_size": 140,
        "ttl": 300,
        "answer_count": 1,
        "additional_count": 0,
        "query_type": qtype,
        "response_type": qtype,
        "has_txt_record": 0,
        "has_null_record": 0,
        "query_entropy": _entropy(compact),
        "label_entropy": float(np.mean([_entropy(label.lower()) for label in labels])) if labels else 0.0,
        "subdomain_count": max(len(labels) - 2, 0),
        "max_label_length": max([len(label) for label in labels], default=0),
        "avg_label_length": float(np.mean([len(label) for label in labels])) if labels else 0.0,
        "unique_chars": len(set(compact)),
        "digit_ratio": sum(ch.isdigit() for ch in compact) / max(len(compact), 1),
        "consonant_ratio": sum(ch in consonants for ch in compact) / max(len(compact), 1),
        "hex_ratio": sum(ch in hex_chars for ch in compact) / max(len(compact), 1),
        "inter_arrival_time": 10000.0,
        "query_rate": 0.10,
        "response_ratio": 0.50,
        "unique_domains": 1,
        "is_recursive": 1,
        "authoritative_flag": 0,
        "retransmission_count": 0,
        "truncated_flag": 0,
        "tcp_stream_id": 0,
        "segment_count": 0,
        "tcp_payload_size": 0,
        "window_size": 0,
        "tcp_flags": 0,
        "retransmission_ratio": 0.0,
        "dns_length_field": 0,
        "message_count": 1,
        "max_message_size": 84,
        "avg_message_size": 70.0,
        "rcode": 0,
    }

sanity_domains = ["ping.archlinux.org", "mirror.archlinux.org", "geo.mirror.pkgbuild.com", "pool.ntp.org"]
sanity_raw = pd.DataFrame(normal_lookup_row(domain) for domain in sanity_domains)
sanity_features = add_derived_features(sanity_raw).reindex(columns=features, fill_value=0)
sanity_proba = best_model.predict_proba(sanity_features)[:, 1]
sanity_eval = pd.DataFrame({
    "query_name": sanity_domains,
    "malicious_probability": sanity_proba,
    "tui_label_at_70pct": np.where(sanity_proba >= DECISION_THRESHOLD, "Malicious", "Benign"),
})
sanity_eval.to_csv(OUTPUT_DIR / "known_good_sanity_check.csv", index=False)
sanity_eval

,query_name,malicious_probability,tui_label_at_70pct
0,ping.archlinux.org,0.514255,Benign
1,mirror.archlinux.org,0.517842,Benign
2,geo.mirror.pkgbuild.com,0.537802,Benign
3,pool.ntp.org,0.517112,Benign


<a id="9"></a>
## 9. Artifact Export

The TUI expects classic DNS artifacts in `artifacts/classic_dns/`. This section saves the trained models, feature names, model comparison tables, threshold analysis, confusion matrices, classification reports, feature importance files, and metadata used by deployment.

In [11]:
def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")

def _get_output_features(model: Pipeline, input_features: list[str]) -> list[str]:
    preprocess = model.named_steps.get("preprocess")
    if hasattr(preprocess, "get_feature_names_out"):
        try:
            return preprocess.get_feature_names_out().tolist()
        except Exception:
            pass
    return input_features

def save_importance(models: dict[str, Pipeline], features: list[str], output_dir: Path) -> None:
    rf = models["random_forest"]
    rf_features = _get_output_features(rf, features)
    pd.DataFrame({"feature": rf_features, "importance": rf.named_steps["model"].feature_importances_}).sort_values(
        "importance", ascending=False
    ).to_csv(output_dir / "random_forest_feature_importance.csv", index=False)

    xgb = models["xgboost"]
    xgb_features = _get_output_features(xgb, features)
    pd.DataFrame({"feature": xgb_features, "importance": xgb.named_steps["model"].feature_importances_}).sort_values(
        "importance", ascending=False
    ).to_csv(output_dir / "xgboost_feature_importance.csv", index=False)

    lr = models["logistic_regression"]
    lr_features = _get_output_features(lr, features)
    pd.DataFrame({"feature": lr_features, "coefficient": lr.named_steps["model"].coef_[0]}).assign(
        abs_coefficient=lambda frame: frame["coefficient"].abs()
    ).sort_values("abs_coefficient", ascending=False).to_csv(
        output_dir / "logistic_regression_coefficients.csv", index=False
    )

metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
threshold_df.to_csv(OUTPUT_DIR / "threshold_analysis.csv", index=False)
pd.Series(features, name="feature").to_csv(OUTPUT_DIR / "feature_names.csv", index=False)
save_importance(models, features, OUTPUT_DIR)
save_json(OUTPUT_DIR / "classification_reports.json", full_metrics)
save_json(OUTPUT_DIR / "confusion_matrices.json", matrices)
save_json(
    OUTPUT_DIR / "model_metadata.json",
    {
        "dataset": str(DATASET),
        "target": TARGET,
        "best_model": best_model_name,
        "random_state": RANDOM_STATE,
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "feature_count": len(features),
        "label_mapping": {"0": "benign", "1": "dns_tunnel"},
        "transport_model": "combined_udp_tcp_classic_dns",
        "decision_threshold": DECISION_THRESHOLD,
        "threshold_metrics": selected_threshold_metrics,
        "tui_transport_routing": {
            "UDP53": "artifacts/classic_dns",
            "TCP53": "artifacts/classic_dns",
            "DOH": "artifacts/doh",
        },
    },
)
save_json(
    OUTPUT_DIR / "data_profile.json",
    {
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "label_counts": df[TARGET].value_counts().sort_index().to_dict(),
        "transport_counts": df["transport"].value_counts().to_dict() if "transport" in df else {},
        "top_level_counts": df["top_level"].value_counts().to_dict() if "top_level" in df else {},
    },
)
print(f"Artifacts saved to {OUTPUT_DIR}")

Artifacts saved to /home/gyro/Documents/Projects/cns_projects/c1-dns-tunneling/artifacts/classic_dns


<a id="10"></a>
## 10. TUI Deployment Mapping

The live/replay monitor is transport-aware:

| Transport | Artifact folder | Behavior |
|---|---|---|
| UDP53 | `artifacts/classic_dns/` | Classic DNS model prediction |
| TCP53 | `artifacts/classic_dns/` | Classic DNS model prediction |
| DOH | `artifacts/doh/` | Separate DoH model prediction |

If an artifact folder is missing, the TUI displays `Unsupported model` instead of producing a placeholder prediction.

In [12]:
deployment_routing = pd.DataFrame([
    {"transport": "UDP53", "artifact_dir": "artifacts/classic_dns", "model_file": "best_model.pkl", "threshold": DECISION_THRESHOLD},
    {"transport": "TCP53", "artifact_dir": "artifacts/classic_dns", "model_file": "best_model.pkl", "threshold": DECISION_THRESHOLD},
    {"transport": "DOH", "artifact_dir": "artifacts/doh", "model_file": "xgboost.pkl", "threshold": DECISION_THRESHOLD},
])
deployment_routing["ready"] = deployment_routing.apply(
    lambda row: (Path(row["artifact_dir"]) / row["model_file"]).exists(), axis=1
)
print(f"Replay command: .venv/bin/python pipeline/tui.py --mode replay --malicious-threshold {DECISION_THRESHOLD:.2f}")
deployment_routing

Replay command: .venv/bin/python pipeline/tui.py --mode replay --malicious-threshold 0.70


,transport,artifact_dir,model_file,threshold,ready
0,UDP53,artifacts/classic_dns,best_model.pkl,0.7,False
1,TCP53,artifacts/classic_dns,best_model.pkl,0.7,False
2,DOH,artifacts/doh,xgboost.pkl,0.7,False


## Optional: Feature Importance Inspection

The feature-importance table is useful for explaining model behavior in the report or class demo. For example, high importance on query entropy, long labels, TXT/NULL usage, and query-rate features aligns with how DNS tunneling tools encode and move data.

In [13]:
importance_path = OUTPUT_DIR / "random_forest_feature_importance.csv"
if importance_path.exists():
    pd.read_csv(importance_path).head(20)
else:
    print("Run the save-artifacts cell first.")